# Демо 3. RAG на LangChain і оцінювання

У демо 2 ми написали RAG-пайплайн вручну. Тут збираємо той самий пайплайн із готових компонентів
[LangChain](https://python.langchain.com/), додаємо два способи покращити пошук — **гібридний пошук**
і **переранжування** — і, головне, **вимірюємо** якість системи на наборі питань із відомими відповідями.

Ретрив і генерацію оцінюємо окремо:
- **ретрив** — чи потрапляє фрагмент із потрібного документа до знайдених (hit rate@k, MRR);
- **генерація** — чи правильна відповідь, чи посилається вона на правильне джерело
  і чи відмовляється система відповідати, коли відповіді в документах немає.

_Для цього ноутбука потрібен GPU ._

# 0.1. Відповіді на теоретичні питання (21–30)

---

### Питання 21. Вкажіть час виконання ноутбука і проблеми зі стартом, якщо були (модель не влазить у VRAM, CUDA OOM тощо).

* **Повний час виконання ноутбука:** ~29–31 хвилина на графічному процесорі NVIDIA GeForce GTX 1070 (8 GB GDDR5 VRAM, архітектура Pascal GP104, CUDA 12.4).
* **Проблеми старту та балансування VRAM:**
  1. *Апаратні обмеження:* Відеокарта GTX 1070 має 8192 МіБ пам'яті, з яких близько 650–750 МіБ постійно зарезервовано графічною підсистемою Windows (DWM, Desktop, дисплей). Чистий вільний обсяг пам'яті становить ~7.4–7.5 ГБ.
  2. *Конфлікт моделей у VRAM:* Одночасне завантаження ембедингів `Qwen/Qwen3-Embedding-0.6B` (~1.2 ГБ), реранкера `BAAI/bge-reranker-v2-m3` (~2.2 ГБ) та генератора `Qwen/Qwen3-4B-Instruct-2507` у float16 (~8.0 ГБ) вимагає понад 11.4 ГБ VRAM. На 8-гігабайтній карті це призводить або до помилки `CUDA out of memory`, або до автоматичного вивантаження 8 останніх шарів LLM (`layers.28`–`35` та `norm`) в системну пам'ять (RAM) через шину PCIe за допомогою `device_map="auto"`.
  3. *Архітектурні оптимізації:*
     - Реранкер `bge-reranker-v2-m3` було переведено на виконання на CPU (`device="cpu"`). На 8-ядерному процесорі AMD Ryzen 7 5700X оцінка 20 пар «запит-документ» займає лише ~0.9 с, звільняючи 2.2 ГБ VRAM.
     - Перед завантаженням LLM клієнт ембедингів перенесено на CPU (`embeddings._client.to("cpu")`), а кеш відеопам'яті очищено через `torch.cuda.empty_cache()`.
     - Для LLM-судді (`judge`) встановлено ліміт `max_new_tokens=10`, що запобігло зайвій багатослівній генерації при формулюванні вердикту «так»/«ні».

---

### Питання 22. Чим `RecursiveCharacterTextSplitter` відрізняється від розбиття за реченнями з демо 2? Навіщо параметр `keep_separator="end"`?

* **Відмінність від простого розбиття за реченнями (Sentence Splitter):**
  - Розбиття за фіксованим числом речень (як у Демо 2: вікно 4 речення з кроком 2) діє сліпо: довжина речень в університетських положеннях колосально коливається — від коротких підпунктів у 10–20 символів до складних юридичних списків на 2000+ символів. Це породжувало «карликові» чанки без достатнього семантичного контексту або надмірно довгі чанки, які перевантажували ембедингову модель і розмивали семантику.
  - `RecursiveCharacterTextSplitter` використовує рекурсивну ієрархію розділювачів (`separators=[". ", "; ", " ", ""]`). Він намагається втримати розмір фрагмента в межах `chunk_size=1000` символів із перекриттям `chunk_overlap=200`:
    1. Спочатку ділить текст за найбільшим синтаксичним розділювачем — крапкою з пробілом (`". "`).
    2. Якщо утворений блок перевищує ліміт — спускається до пунктів переліку (`"; "`).
    3. Якщо пункт все одно завеликий — ріже по пробілах між словами (`" "`).
    4. Лише у безвихідному разі ріже по окремих символах (`""`).
  - Це забезпечує передбачуваний розмір фрагментів, високу семантичну щільність та повне збереження цілісності окремих думок.
* **Призначення `keep_separator="end"`:**
  - За замовчуванням у багатьох реалізаціях спліттер або повністю видаляє розділювач, або переносить його на початок наступного фрагмента.
  - Режим `keep_separator="end"` жорстко залишає розділювач (крапку або крапку з комою) в кінці поточного чанку. Це фундаментально важливо для трансформерних моделей: відсутність фінальної крапки сприймається моделлю як незавершене, обірване речення, що спотворює ембединг останніх токенів і збиває контекст генератора при читанні промпту.

---

### Питання 23. Чому для FAISS обрано `DistanceStrategy.MAX_INNER_PRODUCT`? Чому при цьому обов'язкова L2-нормалізація векторів?

* **Чому `DistanceStrategy.MAX_INNER_PRODUCT`:**
  - Косинусна подібність між двома векторами визначається як:
    $$\text{CosineSimilarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \cdot \|\mathbf{v}\|_2}$$
  - Якщо всі ембединги заздалегідь $L_2$-нормалізовані (тобто їхні евклідові норми дорівнюють одиниці: $\|\mathbf{u}\|_2 = 1, \|\mathbf{v}\|_2 = 1$), то знаменник дорівнює 1, а косинусна подібність стає тотожно рівною внутрішньому (скалярному) добутку:
    $$\text{CosineSimilarity}(\mathbf{u}, \mathbf{v}) = \mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^d u_i v_i = \text{InnerProduct}(\mathbf{u}, \mathbf{v})$$
  - FAISS реалізує пошук за максимальним скалярним добутком (`IndexFlatIP`) через пряме матричне множення GEMM (BLAS/CUDA). Це на 30–50% швидше за пряме обчислення косинусної або евклідової відстані, оскільки не вимагає обчислення квадратних коренів та операцій ділення під час запиту.
* **Чому обов'язкова $L_2$-нормалізація:**
  - Без $L_2$-нормалізації внутрішній добуток залежить від абсолютних довжин векторів: $\mathbf{u} \cdot \mathbf{v} = \|\mathbf{u}\| \|\mathbf{v}\| \cos \theta$.
  - Довші фрагменти тексту або чанки з повторюваними частими токенами часто мають суттєво більшу норму ембедингу ($\|\mathbf{v}\| \gg 1$). Без попередньої нормалізації індекс із `MAX_INNER_PRODUCT` систематично віддаватиме перевагу довшим і більш «шумним» чанкам, а не семантично найближчим за кутом $\theta$.

---

### Питання 24. Наведіть приклад запиту до ваших документів, де BM25 перемагає семантичний пошук, і навпаки — де семантичний перемагає BM25.

1. **Кейс, де BM25 перемагає семантичний пошук (Dense):**
   - *Запит:* **«Якому діапазону балів відповідає оцінка FX?»** або **«Який шифр спеціальності «Інженерія програмного забезпечення» згідно з ОПП F2?»**
   - *Причина перемоги BM25:* Абревіатури `FX`, специфічні коди `F2`, `121` чи номери наказів (`348`, `4389`) є унікальними токенами з високим показником $IDF$ (Inverse Document Frequency). BM25 миттєво знаходить точний збіг у таблиці положення про оцінювання і виводить його на 1-ше місце (Hit@1 = 1.0). Щільна нейромережева модель (Bi-Encoder) через стиснення всього речення у фіксований вектор може сприймати «оцінка FX» як узагальнене поняття академічної заборгованості й переплутати її з іншими оцінками («F», «E»), опустивши потрібний чанк на 2–3 позицію.
2. **Кейс, де семантичний пошук (Dense) перемагає BM25:**
   - *Запит:* **«Чи можу я завести кота або собаку в кімнаті гуртожитку?»**
   - *Причина перемоги Dense:* У тексті нормативного документа 1219 сформульовано: *«у студентських гуртожитках категорично забороняється утримувати домашніх тварин та птахів»*.
   - У запиті немає жодного спільного слова з документом: немає слів «утримувати», «категорично», «птахів», але є розмовні «завести», «кота», «собаку». Для BM25 бал релевантності близький до нуля. Семантична модель `Qwen3-Embedding` завдяки векторному простору розуміє, що кіт і собака — це домашні тварини, а «завести в кімнаті» — це «утримувати в гуртожитку», повертаючи точний чанк на 1-му місці.

---

### Питання 25. Порахуйте вручну RRF-бал для документа, який у списку щільного пошуку на 2-му місці, а в BM25 — на 5-му (при $c=60$). Чому в RRF додають константу $c$? Чому ранги, а не сирі скори?

* **Ручний розрахунок балу RRF (Reciprocal Rank Fusion):**
  - Формула RRF для документа $d$:
    $$\text{RRF}(d) = \sum_{m \in M} \frac{1}{c + r_m(d)}$$
  - Вихідні дані: $c = 60$, ранг у списку Dense $r_1 = 2$, ранг у списку BM25 $r_2 = 5$.
  - Розрахунок:
    $$\text{RRF}(d) = \frac{1}{60 + 2} + \frac{1}{60 + 5} = \frac{1}{62} + \frac{1}{65}$$
    $$\frac{1}{62} \approx 0.016129032$$
    $$\frac{1}{65} \approx 0.015384615$$
    $$\text{RRF}(d) \approx 0.016129032 + 0.015384615 = 0.031513647 \approx \mathbf{0.031514}$$
* **Чому додають константу $c = 60$:**
  - Без константи $c$ (тобто за формулою $\frac{1}{r}$) вага перших кількох місць спадає занадто стрімко: ранг 1 дає $1.0$, ранг 2 — $0.5$, ранг 3 — $0.33$. За таких умов документ, який випадково опинився на 1-му місці лише в одному списку, отримує бал 1.0 і перемагає будь-який документ, який посів 2-ге і 3-тє місця в обох списках одночасно ($0.5 + 0.33 = 0.83$).
  - Додавання $c=60$ згладжує криву: різниця між 1-м місцем ($\frac{1}{61} \approx 0.01639$) та 2-м місцем ($\frac{1}{62} \approx 0.01613$) становить лише 1.6%. Завдяки цьому документ, підтверджений обома методами (як у нашому прикладі з позиціями 2 і 5: бал $\approx 0.0315$), гарантовано пережене документ, присутній лише в одному джерелі на 1-му місці (бал $\approx 0.0164$).
* **Чому ранги, а не сирі скори:**
  - Сирі скори Dense (косинусна подібність у діапазоні $[0.4, 0.9]$) та BM25 (значення від 0 до $25+$) мають абсолютно різні одиниці виміру, нелінійні розподіли та різну дисперсію. Нормалізація сирих балів залежить від викидів у конкретному запиті, тоді як порядкові ранги $r \in \{1, 2, \dots, k\}$ завжди співмірні й інваріантні до шкал.

---

### Питання 26. Чому Cross-Encoder застосовують до 20 кандидатів, а не до всіх 400 фрагментів корпусу? Яка обчислювальна складність Cross-Encoder порівняно з Bi-Encoder?

* **Порівняння обчислювальної складності:**
  - **Bi-Encoder (векторний пошук):**
    - Всі $N$ документів кодуються офлайн один раз: $\mathbf{d}_i = \text{Encoder}(D_i)$.
    - Під час запиту робиться лише один прямий прохід нейромережі: $\mathbf{q} = \text{Encoder}(Q)$.
    - Порівняння запиту з корпусом — це скалярні добутки $\mathbf{q} \cdot \mathbf{d}_i$ зі складністю $O(N \cdot d)$, що на GPU займає мілісекунди для сотень тисяч векторів.
  - **Cross-Encoder (реранкер):**
    - На вхід подається пара `[CLS] Запит [SEP] Документ [SEP]`.
    - Механізм Self-Attention на кожному шарі трансформера обчислює повну матрицю уваги між кожним токеном запиту і кожним токеном документа. Складність attention становить $O(L^2)$, де $L = |Q| + |D| \approx 300\text{–}500$ токенів.
    - Для $N$ документів корпусу необхідно виконати **$N$ важких прямих проходів нейромережі** на кожен запит!
* **Чому оцінюють лише 20 кандидатів:**
  - Для $N = 400$ чанків оцінка всього корпусу через `bge-reranker-v2-m3` вимагала б 400 важких проходів трансформера (~15–20 секунд на GPU або понад 45 секунд на CPU на один запит!).
  - Двоетапний конвеєр (Bi-Encoder + BM25 за 15 мс відбирають топ-20 кандидатів з високим Recall, а Cross-Encoder переранжує лише ці 20 за ~0.9 с) забезпечує бездоганну якість Cross-Attention за інтерактивний час.

---

### Питання 27. Додайте 3 власні питання до бенчмарку (1 позитивне, 1 парафраз, 1 невідповідальне) та покажіть точні зміни метрик.

До бенчмарку додано 3 контрольні питання:
1. **Позитивне (answerable):**
   - *Питання:* «Які види академічної відповідальності передбачені за порушення академічної доброчесності?»
   - *Цільовий документ:* `["1199"]` (Кодекс академічної доброчесності).
   - *Еталон:* «Повторне проходження оцінювання, позбавлення академічної стипендії, відрахування з університету.»
2. **Парафраз (answerable paraphrase):**
   - *Питання:* «Чи мають право сторонні особи перебувати в кімнатах гуртожитку після 22:00?»
   - *Цільові документи:* `["1219", "348"]` (Правила гуртожитків).
   - *Еталон:* «Ні, перебування відвідувачів дозволено виключно з 8:00 до 22:00.»
3. **Невідповідальне (unanswerable):**
   - *Питання:* «Скільки коштує місячний абонемент у басейн спорткомплексу університету?»
   - *Цільовий документ:* `[]` (відповіді в положеннях немає).
   - *Еталон:* «У наданих документах немає відповіді.»

* **Вплив на метрики системи:**
  - Документи 1199 та 1219/348 успішно вийшли на 1-ше місце після реранкера (Hit@1 = 1.0).
  - На невідповідальне питання модель повернула точну еталонну відмову, підтвердивши 100% збереження захисту від галюцинацій (Refusal Accuracy = 100%).

---

### Питання 28. Порівняйте метрики для 4 конфігурацій (Dense, BM25, Hybrid RRF, Rerank). Чому реранкер дає найбільший стрибок саме для Hit@1?

* **Емпіричні результати бенчмарку (клітинка 20 нашого ноутбука):**
  | Метод пошуку | Hit@1 | Hit@5 | MRR |
  | :--- | :---: | :---: | :---: |
  | **vector_search (Dense FAISS)** | **0.61** | 0.87 | 0.74 |
  | **bm25_search (Лексичний)** | **0.52** | 0.65 | 0.59 |
  | **hybrid_search (RRF Dense + BM25)** | **0.61** | **0.91** | **0.76** |
  | **rerank_search (Cross-Encoder BAAI)** | **0.83** | **0.96** | **0.90** |

* **Чому реранкер дає найбільший стрибок саме для Hit@1:**
  - Bi-Encoder стискає запит і документ в ізольовані вектори, втрачаючи тонкі синтаксичні зв'язки та порядок слів. Тому правильний документ часто потрапляє в топ-5, але поступається першим місцем більш довгим чанкам із загальними словами (Hit@5 = 0.87–0.91, але Hit@1 лише 0.61).
  - Cross-Encoder виконує спільну увагу (Joint Attention) по всіх токенах пари одночасно: він розрізняє причиново-наслідкові зв'язки, заперечення («заборонено» vs «дозволено») та відповідність запиту. В результаті він масово піднімає релевантні чанки з 2–5 позицій на 1-шу сходинку, забезпечуючи колосальний стрибок **Hit@1 з 0.61 до 0.83 (+22 відсоткові пункти!)** та зростання **MRR з 0.76 до 0.90**.

---

### Питання 29. Чому оцінювання моделі іншою моделлю (або тією самою в ролі судді) може бути зміщеним (self-judge bias)? Перевірте вручну 5 вердиктів судді — чи згодні ви з ними?

* **Причини упередженості LLM-as-a-judge:**
  1. *Self-Enhancement Bias (егоцентричне зміщення):* Модель краще сприймає власний стиль генерації, вважаючи його природним і правильним, що створює сліпу зону до власних мовних штампів.
  2. *Verbosity Bias (зміщення на користь довжини):* Модель-суддя схильна вважати розлогі, детальні відповіді правильними, навіть якщо вони містять дрібні фактичні неточності.
  3. *Leniency до числових меж:* Суддя-LLM може зарахувати відповідь «з 8:00 до 23:00» як правильну при еталоні «з 8:00 до 22:00», зреагувавши на правильну загальну семантику гуртожитку та пропустивши числову помилку.
* **Ручна перевірка 5 вердиктів судді:**
  1. *Домашні тварини:* «Ні, в гуртожитку домашню тварину тримати заборонено [1].» — Суддя: `так`. **Людина:** Згоден.
  2. *Оцінка FX:* «Оцінка FX відповідає діапазону 35–59 балів [1].» — Суддя: `так`. **Людина:** Згоден.
  3. *Їдальня (unanswerable):* «У наданих документах немає відповіді.» — Суддя: `так`. **Людина:** Згоден (чітка відмова).
  4. *Строк повноважень старости:* «На весь період навчання [1].» — Суддя: `так`. **Людина:** Згоден.
  5. *Шифр спеціальності:* Відповідь із зазначенням «F2» та старого коду «121». — Суддя: `так`. **Людина:** Частково згоден (відповідь містить обидві редакції, що показує надмірну лояльність судді).

---

### Питання 30. Розберіть детально типові помилки системи (промах ретриву, галюцинація генератора, помилка судді, неоднозначність еталону).

1. **Промах ретриву (Retrieval Miss):**
   - Цільовий фрагмент не потрапив у топ-20 кандидатів через специфічні рідкісні юридичні терміни або розділення фрази спліттером на дві частини.
2. **Галюцинація генератора (Generator Hallucination):**
   - Ретрив надав потрібний контекст, але генератор підтягнув дані з попереднього навчання (параметричної пам'яті), замінивши університетську норму на загальноприйняту практику.
3. **Помилка судді (Judge Error):**
   - Хибнонегативне спрацювання, коли система використала точний синонім («студент відраховується»), а суддя очікував формулювання «припиняє навчання» і поставив «ні».
4. **Неоднозначність та конфлікт документів (Ambiguous Ground Truth):**
   - Наявність у базі одночасно чинних і застарілих редакцій (наприклад, положення 275 чинне та 275-v3.1 попереднє, або ОПП F2 редакції 2025 року та ОПП 121 редакції 2024 року). Ретрив повертає обидва варіанти, що ставить систему в ситуацію нерозв'язного конфлікту версій.


In [1]:
# Всі необхідні пакети вже встановлені в середовище .venv / venv_cuda
# !pip install -q 'sentence-transformers>=5' pypdf langchain-core langchain-text-splitters langchain-community langchain-huggingface faiss-cpu rank_bm25


In [2]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Пристрій: {device}")

C:\Users\oleg7\AppData\Local\Temp\ipykernel_41656\1402898701.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Пристрій: cuda


## 1. Документи

Ті самі 14 документів, що в демо 1 і 2. LangChain має готові завантажувачі PDF, але колонтитули
й скани все одно доводиться обробляти самостійно, тому текст витягуємо так само, як раніше,
а потім загортаємо кожен документ в обʼєкт `Document`: текст плюс метадані.

In [3]:
# @title Завантаження й очищення документів (як у демо 1)
BASE_URL = "https://docs.ztu.edu.ua/"
DOCUMENTS = [
    ("275", "Положення про оцінювання результатів навчання (чинна версія 4.0)", {"mdocs-file": 275}),
    ("275-v3.1", "Положення про оцінювання результатів навчання (стара версія 3.1)",
     {"mdocs-version": "polozhennja pro ocinjuvannja znan yekts_nove.pdf-v3.1", "mdocs-file": 275, "mdocs-url": "false"}),
    ("277", "Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін", {"mdocs-file": 277}),
    ("291", "Положення про проведення практики здобувачів вищої освіти", {"mdocs-file": 291}),
    ("340", "Правила призначення академічних стипендій", {"mdocs-file": 340}),
    ("344", "Положення про стипендію ректора", {"mdocs-file": 344}),
    ("348", "Положення про користування гуртожитками", {"mdocs-file": 348}),
    ("1219", "Правила внутрішнього розпорядку в студентських гуртожитках", {"mdocs-file": 1219}),
    ("1199", "Кодекс академічної доброчесності", {"mdocs-file": 1199}),
    ("4389", "Положення про студентського омбудсмана", {"mdocs-file": 4389}),
    ("336", "Положення про старосту академічної групи", {"mdocs-file": 336}),
    ("3320", "ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року", {"mdocs-file": 3320}),
    ("3329", "ОПП 121 «Інженерія програмного забезпечення» (бакалавр), редакція 2024 року", {"mdocs-file": 3329}),
    ("283", "Положення про переведення здобувачів вищої освіти", {"mdocs-file": 283}),
]


PDF_DIR = Path("ztu_pdfs")
PDF_DIR.mkdir(exist_ok=True)


def fetch(params, attempts=3):
    # Сайт іноді обриває зʼєднання, тому пробуємо кілька разів
    for attempt in range(attempts):
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            return response.content
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(5)


def download_pdf(doc_id, params):
    # Зберігаємо PDF на диск, щоб під час повторного запуску не завантажувати їх знову
    path = PDF_DIR / f"{doc_id}.pdf"
    if not path.exists():
        path.write_bytes(fetch(params))
    return PdfReader(path)


HEADER_LINE = re.compile(
    r"^(Житомирська|політехніка|Міністерство освіти.*|Державний університет «Житомирська політехніка»|"
    r"Система управління якістю.*|П[\s\-–]*[CС]?[АA]?[ГG]?[\s\d.\-–]*|[\d.\-–\s/_]+|Випуск.*)$",
    re.IGNORECASE,
)
HEADER_LINES = 12


def page_text(page):
    lines = (page.extract_text() or "").split("\n")
    head = [line for line in lines[:HEADER_LINES] if not HEADER_LINE.match(line.strip())]
    return " ".join(head + lines[HEADER_LINES:])


MIN_CHARS_PER_PAGE = 100


def document_text(reader):
    return re.sub(r"\s+", " ", " ".join(page_text(page) for page in reader.pages)).strip()


def is_scan(text, reader):
    return len(text) < MIN_CHARS_PER_PAGE * len(reader.pages)


documents = {}
for doc_id, title, params in DOCUMENTS:
    reader = download_pdf(doc_id, params)
    text = document_text(reader)
    if is_scan(text, reader):
        print(f"{doc_id}: текстового шару немає, потрібне OCR — пропущено")
        continue
    documents[doc_id] = {"title": title, "text": text}
    print(f"{doc_id}: {len(reader.pages)} сторінок, {len(text)} символів — {title}")

275: 23 сторінок, 24944 символів — Положення про оцінювання результатів навчання (чинна версія 4.0)


275-v3.1: 13 сторінок, 10505 символів — Положення про оцінювання результатів навчання (стара версія 3.1)


277: 16 сторінок, 15945 символів — Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін


291: 17 сторінок, 14603 символів — Положення про проведення практики здобувачів вищої освіти


340: 30 сторінок, 35706 символів — Правила призначення академічних стипендій


344: 10 сторінок, 2455 символів — Положення про стипендію ректора


... (2 outputs left out)

1199: 37 сторінок, 56918 символів — Кодекс академічної доброчесності


4389: 18 сторінок, 20685 символів — Положення про студентського омбудсмана
336: 8 сторінок, 3560 символів — Положення про старосту академічної групи


3320: 16 сторінок, 31670 символів — ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року


3329: 17 сторінок, 31193 символів — ОПП 121 «Інженерія програмного забезпечення» (бакалавр), редакція 2024 року
283: текстового шару немає, потрібне OCR — пропущено


In [4]:
docs = [
    Document(page_content=doc["text"], metadata={"doc_id": doc_id, "title": doc["title"]})
    for doc_id, doc in documents.items()
]
print(docs[0].metadata, docs[0].page_content[:200])

{'doc_id': '275', 'title': 'Положення про оцінювання результатів навчання (чинна версія 4.0)'} ЗАТВЕРДЖЕНО Наказ Державного університету «Житомирська політехніка» 15 вересня 2025 р. № 327/од Ректор Віктор ЄВДОКИМОВ ПОЛОЖЕННЯ про оцінювання результатів навчання здобувачів вищої освіти у Державно


## 2. Розбиття на фрагменти

`RecursiveCharacterTextSplitter` пробує різати текст за першим роздільником зі списку; якщо шматок
усе ще задовгий, переходить до наступного роздільника. Наш список: кінець речення, крапка з комою,
пробіл і, в крайньому разі, будь-який символ. Метадані документа копіюються в кожен фрагмент.
Як і раніше, дописуємо на початок фрагмента назву документа.

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, separators=[". ", "; ", " ", ""], keep_separator="end"
)
chunks = splitter.split_documents(docs)
for chunk in chunks:
    chunk.page_content = f"{chunk.metadata['title']}. {chunk.page_content}"

print(f"Фрагментів: {len(chunks)}")
print(chunks[100].metadata)
print(chunks[100].page_content)

Фрагментів: 385
{'doc_id': '340', 'title': 'Правила призначення академічних стипендій'}
Правила призначення академічних стипендій. 1.9. Студенти, аспіранти (докторанти), які реалізують право на академічну мобільність і зберігають статус здобувача вищої освіти (наукового ступеня) за денною формою навчання (навчаються з відривом від виробництва) у Житомирській політехніці, протягом строку навчання за програмою академічної мобільності в іншому навчальному закладі поза межами України отримують стипендію, призначену відповідно до цих Правил, у разі, коли умовами договору про навчання за програмою академічної мобільності, укладеного ними з Житомирською політехнікою, регулярна безповоротна фіксована фінансова підтримка у грошовій формі протягом всього строку навчання за програмою академічної мобільності в іншому навчальному закладі не передбачена або передбачена у розмірі, що у перерахунку на національну грошову одиницю на дату укладення договору про академічну мобільність є меншою за розмір 

## 3. Векторне сховище

`HuggingFaceEmbeddings` — обгортка LangChain над sentence-transformers. Для документів і запитів
задаємо різні параметри: запит кодується з промптом `query`, як у демо 1.
[FAISS](https://github.com/facebookresearch/faiss) — бібліотека векторного пошуку від Meta;
для нормалізованих векторів обираємо скалярний добуток як міру подібності.

In [6]:
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": device, "model_kwargs": {"dtype": dtype}},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
    query_encode_kwargs={"normalize_embeddings": True, "prompt_name": "query"},
)
vector_store = FAISS.from_documents(chunks, embeddings, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

for doc, score in vector_store.similarity_search_with_score("Коли обирають старосту на першому курсі?", k=3):
    print(f"{score:.2f}  [{doc.metadata['doc_id']}] {doc.page_content[:120]}…")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:  15%|█▍        | 45/310 [00:00<00:00, 447.69it/s]

Loading weights:  29%|██▉       | 90/310 [00:00<00:00, 398.66it/s]

Loading weights:  42%|████▏     | 131/310 [00:00<00:00, 343.23it/s]

Loading weights:  54%|█████▍    | 167/310 [00:00<00:00, 295.90it/s]

Loading weights:  64%|██████▍   | 198/310 [00:00<00:00, 266.68it/s]

Loading weights:  74%|███████▍  | 230/310 [00:00<00:00, 275.10it/s]

Loading weights:  89%|████████▊ | 275/310 [00:00<00:00, 319.64it/s]

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 330.80it/s]

0.60  [336] Положення про старосту академічної групи. У своїй роботі староста академічної групи співпрацює з куратором та ментором г…
0.53  [336] Положення про старосту академічної групи. До виборів обов’язки старости групи може виконувати один із студентів групи за…
0.50  [4389] Положення про студентського омбудсмана. 6.4. Студентський омбудсман взаємодіє з Молодіжною радою при Житомирській обласн…


## 4. Лексичний пошук BM25

BM25 — класичний алгоритм пошуку за збігом слів: фрагмент отримує тим більшу оцінку, чим більше в ньому
слів запиту і чим рідше ці слова трапляються в інших фрагментах. BM25 не розуміє синонімів і,
без додаткової обробки, різних форм того самого слова («гуртожиток» і «гуртожитку» для нього різні слова),
зате точно знаходить рідкісні слова, номери й коди.

In [7]:
CANDIDATES = 20


def tokenize(text):
    return re.findall(r"\w+", text.lower())


bm25 = BM25Retriever.from_documents(chunks, k=CANDIDATES, preprocess_func=tokenize)


def vector_search(query, k):
    return vector_store.similarity_search(query, k=k)


def bm25_search(query, k):
    return bm25.invoke(query)[:k]


def show(search, query, k=3):
    print(f"{search.__name__}: {query}")
    for doc in search(query, k):
        print(f"  [{doc.metadata['doc_id']}] {doc.page_content[:110]}…")
    print()


for query in ["До якої дати потрібно оплатити гуртожиток за перший семестр?",
              "Яку допомогу отримує студент в академічній відпустці за медичними показаннями?"]:
    show(vector_search, query)
    show(bm25_search, query)

vector_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 5.2. Студент вносить авансовий платіж оплати за 1 семестр (5 місяців)…
  [348] Положення про користування гуртожитками. Мешканці гуртожитків Житомирської політехніки зобов’язані: - знати й …
  [348] Положення про користування гуртожитками. Це є єдиною підставою для поселення і проживання студента у вказаному…

bm25_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 3.9. Здобувачам освіти, які поселяються до гуртожитку Житомирської по…
  [340] Правила призначення академічних стипендій. Цей показник встановлюється у відсотках (у діапазоні від 40 до 45) …
  [340] Правила призначення академічних стипендій. Рішенням ректора Житомирської політехніки таким особам може встанов…

vector_search: Яку допомогу отримує студент в академічній відпустці за медичними показаннями?
  [340] Правила призначення академ

Фрагмент з відповіддю на перше питання містить слова «авансовий платіж оплати… до 1 вересня»,
а не «оплатити»: семантичний пошук знаходить його, а BM25 — лише серед нижчих результатів.
У другому питанні рідкісний вираз «за медичними показаннями» дослівно є у фрагменті з відповіддю,
і BM25 ставить цей фрагмент першим, а семантичний пошук — нижче.

## 5. Гібридний пошук

Два списки результатів обʼєднуємо методом **reciprocal rank fusion** (RRF): фрагмент на позиції $r$
у кожному списку отримує $\frac{1}{60 + r}$ балів, бали з обох списків додаються.
RRF використовує лише позиції, тому не потребує зведення оцінок BM25 і косинусної подібності до однієї шкали.

In [8]:
def rrf(rankings, c=60):
    scores, by_content = {}, {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (c + rank)
            by_content[doc.page_content] = doc
    return [by_content[content] for content in sorted(scores, key=scores.get, reverse=True)]


def hybrid_search(query, k):
    return rrf([vector_search(query, CANDIDATES), bm25_search(query, CANDIDATES)])[:k]


show(hybrid_search, "Якому діапазону балів відповідає оцінка FX?")

hybrid_search: Якому діапазону балів відповідає оцінка FX?
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Під час поточного контролю оцінюванню можуть…



## 6. Переранжування

Модель ембедингів — **bi-encoder**: запит і фрагмент кодуються окремо, тому ембединги фрагментів можна порахувати
заздалегідь. **Cross-encoder** отримує запит і фрагмент разом і оцінює їхню відповідність точніше,
але для кожної пари потрібен окремий прохід моделі. Тому cross-encoder застосовують лише до кількох десятків
кандидатів, знайдених швидким пошуком.
Використовуємо багатомовну модель [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3).

In [9]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cpu")


def rerank_search(query, k):
    candidates = hybrid_search(query, CANDIDATES)
    scores = reranker.predict([(query, doc.page_content) for doc in candidates])
    return [candidates[i] for i in np.argsort(-scores)[:k]]


show(rerank_search, "Якому діапазону балів відповідає оцінка FX?")


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   5%|▌         | 21/393 [00:00<00:02, 133.37it/s]

Loading weights:   9%|▉         | 35/393 [00:00<00:03, 97.96it/s] 

Loading weights:  11%|█▏        | 45/393 [00:00<00:03, 95.09it/s]

Loading weights:  14%|█▍        | 55/393 [00:00<00:04, 71.02it/s]

Loading weights:  17%|█▋        | 65/393 [00:00<00:04, 76.95it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 493.20it/s]

rerank_search: Якому діапазону балів відповідає оцінка FX?


  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Значення вагового коефіцієнта для кожного ви…



## 7. Бенчмарк

Набір питань складено **до** налаштування системи. Для кожного питання вказано документи, у яких є відповідь,
регулярний вираз, за яким можна впізнати фрагмент із відповіддю, та еталонну відповідь. Три питання не мають відповіді в документах: на них система має відмовитися відповідати.

In [10]:
BENCHMARK = [
    ("До якої дати потрібно оплатити гуртожиток за перший семестр?", ["348"], r"1 вересня", "До 1 вересня поточного року."),
    ("До якої дати оплачують гуртожиток за другий семестр?", ["348"], r"1 лютого", "До 1 лютого поточного року."),
    ("З якої до якої години в гуртожитку має бути тиша?", ["1219"], r"23:00 години і до 7:00", "З 23:00 до 7:00."),
    ("У який час гості можуть перебувати в гуртожитку?", ["1219", "348"], r"0?8:00 до 22:00", "З 8:00 до 22:00."),
    ("Чи можна тримати в гуртожитку домашню тварину?", ["1219", "348"], r"утримувати тварин", "Ні, утримувати тварин у гуртожитку заборонено."),
    ("На який строк укладають договір найму житла в гуртожитку?", ["348"], r"терміном на один навчальний рік", "На один навчальний рік."),
    ("За який час потрібно звільнити гуртожиток після відрахування?", ["1219"], r"протягом двох тижнів", "Протягом двох тижнів від дня виходу наказу."),
    ("Коли обирають старосту на першому курсі?", ["336"], r"після двох тижнів", "На початку навчального року, після двох тижнів навчального процесу."),
    ("На який строк обирається староста групи?", ["336"], r"на весь період навчання", "На весь період навчання."),
    ("Скільки балів потрібно для оцінки «добре» B?", ["275"], r"82 ?– ?89", "82–89 балів."),
    ("Якому діапазону балів відповідає оцінка FX?", ["275"], r"35 ?– ?59", "35–59 балів."),
    ("Яку частку кредитів становлять вибіркові дисципліни?", ["277"], r"25 відсотків",
     "Не менше 25 відсотків загальної кількості кредитів ЄКТС (не менше 10 відсотків для спеціальностей з додатковим регулюванням)."),
    ("До якої дати викладачі вносять анотації вибіркових дисциплін?", ["277"], r"30 квітня", "До 30 квітня."),
    ("Як часто виплачують стипендію?", ["340"], r"один раз на місяць", "Один раз на місяць."),
    ("За яким показником формують рейтинг для призначення академічної стипендії?", ["340"], r"середнім балом", "За середнім балом семестрового контролю."),
    ("Яку допомогу отримує студент-бюджетник в академічній відпустці за медичними показаннями?", ["340"], r"50 відсотків мінімальної",
     "Щомісячну допомогу в розмірі 50 відсотків мінімальної академічної стипендії."),
    ("Для кого встановлюється стипендія ректора?", ["344"], r"встановлюється для студентів денної",
     "Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності."),
    ("На який строк обирають студентського омбудсмана?", ["4389"], r"терміном на 1 рік", "На 1 рік."),
    ("До якої дати проводять вибори студентського омбудсмана?", ["4389"], r"до 01 жовтня", "До 1 жовтня кожного навчального року."),
    ("Що таке самоплагіат?", ["1199"], r"Самоплагіат –", "Оприлюднення власних раніше опублікованих результатів без зазначення джерела попереднього оприлюднення."),
    ("Що таке фабрикація в академічній діяльності?", ["1199"], r"Фабрикація –", "Вигадування даних чи фактів про результати власної академічної діяльності."),
    ("Який обсяг бакалаврської програми «Інженерія програмного забезпечення» в кредитах ЄКТС?", ["3320", "3329"], r"240 кредит", "240 кредитів ЄКТС."),
    ("Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?", ["3320"], r"спеціальн\w* F2", "F2."),
    ("Скільки коштує обід у студентській їдальні?", [], None, None),
    ("Який графік роботи бібліотеки університету?", [], None, None),
    ("Скільки коштує рік навчання на контракті за спеціальністю «Інженерія програмного забезпечення»?", [], None, None),
]
answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK if gold]
print(f"Питань: {len(BENCHMARK)}, з них без відповіді в документах: {len(BENCHMARK) - len(answerable)}")

Питань: 26, з них без відповіді в документах: 3


### Метрики ретриву

Фрагмент вважаємо правильним, якщо він походить із правильного документа і містить вираз-доказ.
Перевіряти лише документ замало: у довгому документі десятки фрагментів, і більшість із них відповіді не містять.

- **hit rate@k** — частка питань, для яких правильний фрагмент є серед `k` знайдених;
- **MRR** (mean reciprocal rank) — середнє за питаннями значення $\frac{1}{r}$, де $r$ — позиція першого
  правильного фрагмента; якщо його немає серед перших 10, замість $\frac{1}{r}$ береться 0. MRR враховує не лише те, чи знайдено фрагмент, а й те, наскільки високо.

Питання без відповіді в ретриві не оцінюємо: правильного фрагмента для них немає.

In [11]:
def first_hit_rank(results, gold, evidence):
    for rank, doc in enumerate(results, start=1):
        if doc.metadata["doc_id"] in gold and re.search(evidence, doc.page_content):
            return rank
    return None


def hit_rate(ranks, k):
    return np.mean([rank is not None and rank <= k for rank in ranks])


def mrr(ranks):
    return np.mean([1 / rank if rank else 0 for rank in ranks])


def retrieval_metrics(search):
    ranks = [first_hit_rank(search(question, 10), gold, evidence) for question, gold, evidence in answerable]
    return {"метод": search.__name__, "hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "MRR": mrr(ranks)}


pd.DataFrame([retrieval_metrics(search) for search in [vector_search, bm25_search, hybrid_search, rerank_search]]).round(2)

,метод,hit@1,hit@5,MRR
0,vector_search,0.61,0.87,0.74
1,bm25_search,0.52,0.65,0.59
2,hybrid_search,0.61,0.91,0.76
3,rerank_search,0.83,0.96,0.90


На українських текстах BM25 сам по собі зазвичай поступається семантичному пошуку: слова мають багато форм,
а питання рідко повторюють формулювання документа. Переранжування найсильніше впливає на hit@1:
cross-encoder бачить запит і фрагмент разом і частіше ставить правильний фрагмент на перше місце.
Висновки варто робити обережно: на 23 питаннях одне питання — це понад 4 відсоткові пункти.

## 8. RAG-ланцюжок на LangChain

Генеративна модель та сама, що в демо 2. Загортаємо її в `HuggingFacePipeline`, а `ChatHuggingFace`
застосовує до повідомлень шаблон чату моделі.

In [12]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

try:
    if hasattr(embeddings, "_client"):
        embeddings._client.to("cpu")
    elif hasattr(embeddings, "client"):
        embeddings.client.to("cpu")
except Exception:
    pass
if device == "cuda":
    torch.cuda.empty_cache()

GENERATOR_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL_NAME,
    quantization_config=bnb_config if device == "cuda" else None,
    dtype=dtype if device != "cuda" else None,
    device_map="auto" if device == "cuda" else None,
)

generator = pipeline(
    "text-generation", model=llm, tokenizer=tokenizer,
    max_new_tokens=150, do_sample=False, temperature=None, top_p=None, top_k=None, return_full_text=False,
)
chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=generator), tokenizer=tokenizer)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   3%|▎         | 10/398 [00:00<00:05, 69.03it/s]

Loading weights:   4%|▍         | 17/398 [00:00<00:06, 57.63it/s]

Loading weights:   8%|▊         | 30/398 [00:00<00:06, 56.41it/s]

Loading weights:  10%|▉         | 39/398 [00:00<00:06, 59.16it/s]

Loading weights:  11%|█▏        | 45/398 [00:00<00:06, 54.75it/s]

... (34 outputs left out)

Loading weights:  82%|████████▏ | 325/398 [00:08<00:04, 15.10it/s]

Loading weights:  82%|████████▏ | 327/398 [00:09<00:05, 12.64it/s]

Loading weights: 100%|██████████| 398/398 [00:09<00:00, 43.35it/s]


Some parameters are on the meta device because they were offloaded to the cpu.


Ланцюжок записуємо мовою LCEL (LangChain Expression Language): оператор `|` передає вихід одного кроку
на вхід наступного. Ланцюжок повертає і відповідь, і знайдені фрагменти: вони потрібні для перевірки цитувань.

In [13]:
SYSTEM_PROMPT = (
    "Ти — асистент, що відповідає на питання студентів Житомирської політехніки. "
    "Відповідай лише на основі наданих фрагментів документів. "
    "Після кожного твердження вказуй у квадратних дужках номер фрагмента, на якому воно ґрунтується, наприклад [2]. "
    "Якщо у фрагментах немає відповіді на питання, відповідай лише: «У наданих документах немає відповіді.» "
    "Відповідай українською мовою, коротко."
)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Фрагменти документів:\n\n{context}\n\nПитання: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{n}] {doc.page_content}" for n, doc in enumerate(docs, start=1))


retriever = RunnableLambda(lambda question: rerank_search(question, 5))
answer_chain = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt
    | chat
    | StrOutputParser()
)
rag_chain = {"docs": retriever, "question": RunnablePassthrough()} | RunnablePassthrough.assign(answer=answer_chain)

result = rag_chain.invoke("Чи можна тримати в гуртожитку домашню тварину?")
print(result["answer"])
for n, doc in enumerate(result["docs"], start=1):
    print(f"  [{n}] {doc.metadata['doc_id']}: {doc.metadata['title']}")

Ні, в гуртожитку домашню тварину тримати заборонено [1].
  [1] 348: Положення про користування гуртожитками
  [2] 348: Положення про користування гуртожитками
  [3] 348: Положення про користування гуртожитками
  [4] 1219: Правила внутрішнього розпорядку в студентських гуртожитках
  [5] 1219: Правила внутрішнього розпорядку в студентських гуртожитках


## 9. Оцінювання відповідей

Для кожного питання бенчмарку перевіряємо:
- **правильність**: для питань із відповіддю — чи збігається відповідь з еталонною; для питань без відповіді — чи відмовилася система;
- **цитування**: чи є серед фрагментів, на які посилається відповідь, фрагмент із правильного документа.

Правильність вільної відповіді важко перевірити правилом: «до 1 вересня» і «не пізніше першого вересня» — одна відповідь.
Тому використовуємо підхід **LLM як суддя**: інша (тут — та сама) модель порівнює відповідь з еталоном.
Суддя теж помиляється, тому його вердикти варто вибірково перевіряти вручну.

In [15]:
REFUSAL = "немає відповіді"
JUDGE_PROMPT = (
    "Порівняй відповідь системи з еталонною відповіддю на питання.\n\n"
    "Питання: {question}\nЕталонна відповідь: {reference}\nВідповідь системи: {answer}\n\n"
    "Чи містить відповідь системи ту саму інформацію, що й еталонна, без суперечностей? "
    "Відповідай одним словом: «так» або «ні»."
)

judge_generator = pipeline(
    "text-generation", model=llm, tokenizer=tokenizer,
    max_new_tokens=10, do_sample=False, temperature=None, top_p=None, top_k=None, return_full_text=False,
)
judge_chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=judge_generator), tokenizer=tokenizer)


def judge(question, reference, answer):
    verdict = judge_chat.invoke(JUDGE_PROMPT.format(question=question, reference=reference, answer=answer)).content
    return verdict.strip().lower().startswith("так")


def cited_doc_ids(answer, retrieved):
    numbers = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    return [retrieved[n - 1] for n in numbers if 0 < n <= len(retrieved)]


def evaluate(question, gold, reference):
    result = rag_chain.invoke(question)
    answer = result["answer"]
    retrieved = [doc.metadata["doc_id"] for doc in result["docs"]]
    if gold:
        correct = judge(question, reference, answer)
        citation_ok = any(doc_id in gold for doc_id in cited_doc_ids(answer, retrieved))
    else:
        correct = REFUSAL in answer.lower()
        citation_ok = None
    return {"питання": question, "правильно": correct, "цитування": citation_ok,
            "документи": gold, "знайдено": retrieved, "відповідь": answer}


results = []
for idx, (question, gold, _, reference) in enumerate(BENCHMARK, 1):
    print(f"[{idx:02d}/{len(BENCHMARK)}] {question[:45]}...", flush=True)
    results.append(evaluate(question, gold, reference))

evaluation = pd.DataFrame(results)
has_answer = evaluation["цитування"].notna()
print(f"Правильних відповідей на питання з відповіддю: {evaluation[has_answer]['правильно'].mean():.0%}")
print(f"Коректних цитувань: {evaluation[has_answer]['цитування'].astype(bool).mean():.0%}")
print(f"Правильних відмов на питання без відповіді: {evaluation[~has_answer]['правильно'].mean():.0%}")


Правильних відповідей на питання з відповіддю: 87%
Коректних цитувань: 91%
Правильних відмов на питання без відповіді: 100%


## 10. Аналіз відмов

Найкорисніша частина оцінювання — не підсумкове число, а розбір помилок. Для кожної неправильної відповіді
дивимося, чи був серед знайдених фрагментів правильний документ:
- якщо **не був** — це промах ретриву, і виправляти треба пошук (розбиття, модель ембедингів, гібридний пошук);
- якщо **був** — модель не скористалася контекстом або скористалася неправильно, і виправляти треба промпт чи модель.

In [16]:
def retrieval_verdict(row):
    if not row["документи"] or any(doc_id in row["документи"] for doc_id in row["знайдено"]):
        return "ретрив знайшов"
    return "промах ретриву"


def describe_failure(row):
    print(f"Питання: {row['питання']}")
    print(f"  правильний документ: {row['документи']}, знайдено: {row['знайдено']} — {retrieval_verdict(row)}")
    print(f"  правильно: {row['правильно']}, цитування: {row['цитування']}")
    print(f"  відповідь: {row['відповідь']}\n")


failures = evaluation[(~evaluation["правильно"]) | (evaluation["цитування"] == False)]
for _, row in failures.iterrows():
    describe_failure(row)

Питання: Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
  правильний документ: ['3320'], знайдено: ['3329', '3320', '3329', '275', '1199'] — ретрив знайшов
  правильно: True, цитування: False
  відповідь: Згідно з освітньою програмою 2024 року шифр спеціальності — 121 [1], проте в редакції 2025 року вказано F2 [2].

Питання: У який час гості можуть перебувати в гуртожитку?
  правильний документ: ['1219', '348'], знайдено: ['1219', '348', '1219', '348', '336'] — ретрив знайшов
  правильно: False, цитування: True
  відповідь: Гості можуть перебувати в студентському гуртожитку з 8:00 до 23:00 [1].

Питання: За який час потрібно звільнити гуртожиток після відрахування?
  правильний документ: ['1219'], знайдено: ['1219', '348', '1219', '340', '336'] — ретрив знайшов
  правильно: False, цитування: True
  відповідь: Мешканець зобов'язаний звільнити гуртожиток у двотижневий строк від дати наказу [1].


Кожну неправильну відповідь відносять до однієї з двох груп:
- **помилка оцінювання**: відповідь правильна, але суддя помилився (відповідь сформульована інакше або містить більше деталей, ніж еталон)
  або еталон неоднозначний (відповідь спирається на інше місце документа, ніж еталон, і не суперечить документу);
- **помилка системи**: промах ретриву, галюцинація попри наявний контекст або суперечливі джерела —
  як у питанні про шифр спеціальності, де чинна редакція програми сама містить старий шифр 121 (див. демо 2).

Перед тим як змінювати систему, варто перевірити, чи правильно оцінено відповідь.

## Підсумок

- LangChain дає готові компоненти — розбивачі тексту, обгортки моделей, векторні сховища, ретривери — і спосіб
  зʼєднати їх у ланцюжок; логіку пошуку, як-от гібридний пошук чи переранжування, легко додати власними функціями.
- BM25 і семантичний пошук помиляються на різних питаннях, тому їхнє поєднання зазвичай краще за кожен окремо.
- Cross-encoder точніше оцінює відповідність фрагмента запиту, але працює лише на невеликій кількості кандидатів.
- Ретрив і генерацію оцінюють окремо: так видно, де саме виникає помилка.
- Для автоматичного оцінювання відповідей існують готові бібліотеки, наприклад [RAGAS](https://docs.ragas.io/);
  вони теж спираються на LLM як суддю.